# **2. Implementation of the solution**

### **2.1.Importation of the libraries that wil be used**

In [ ]:
import math 
import vrplib
import random
import time
import statistics
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt 

### **2.2 Loading one instance for test purposes**

We will begin by importing the file C101.txt once again to test each step of our proposed solution. As said in the previous deliverable this file provides the depot and customers, their coordinates, their demands and time windows along with the service duration, the vehicle capacity and the maximum number of vehicles. All those values will be read by the library to create a distance matrix that we will exploit.

In [ ]:
# We import the file 
instance = vrplib.read_instance("C101.txt", instance_format="solomon")
# Then we will extract the main values that we will use in the algorithm 
D = instance["edge_weight"]          # Distances between locations
n = len(instance["demand"]) - 1      # Number of customers, excluding the depot
m = int(instance["vehicles"])        # Maximum number of available vehicles
Q = float(instance["capacity"])      # Maximum capacity of one vehicle
print("Customers:", n, "Maximum vehicles:", m, "Vehicle capacity:", Q)
print("Distance matrix dimensions:", D.shape)


## **2.3 Granular neighbour list**

The first step of the solution that we proposed is to build a "granular neighbor list" meaning that for each customer we will keep a short list of its closest other customers.

We use this approach so that the local search that we will use later can prioritize promising changing within nearby customers intsead of testing every single pair. Our search will then be more effecient or at least faster.

To be able to do so we will implement a function that uses the parameter "kappa". This parameter represents the number of nearest neighbors that we keep for each customer.

This function get the distance matix from the instance, then loops over every customer i. After that it makes a list of all the other customers j then sort them using their distance from i to finally keep only the first "kappa" customers.

If we use too few neighbours the search will be faster but it may miss useful moves so we choosed to allow a fallback to other customers if no feasible insertion is found.

In [ ]:
def build_neighbours(inst, kappa = 10):
    # Distance matrix : D[i,j] is the distance from i to j 
    distances = inst["edge_weight"]
    # Since our depot is 0, the customers indices start from 1 to n 
    number_of_customers = len(inst["demand"]) - 1
    # We will then store the list of neighbours for each customer in a dictionary 
    neighbours = {}

    #Then we create a loop to visit each customer i in turn : 
    for i in range(1, number_of_customers + 1):
        # We list all other customers; a customer cannot be its own neighbour.
        other_customers = [
            j for j in range(1, number_of_customers + 1)
            if j != i
        ]
        # We sort other customers by increasing distance from i.
        ordered = sorted(other_customers, key=lambda j: distances[i, j])
        # We keep only the first kappa neighbours.
        neighbours[i] = ordered[:kappa]
    return neighbours

# Let's then execute this first step : 
neighbours = build_neighbours(instance, kappa=min(10, n - 1))
print("Neighbours of customer 1:", neighbours[1])

#### **Helper function**

To be able to use a Clarke-Wright algorithm and the rest of our algorithm we need functions that we will use to decide wether it is allowed or not to merge routes for example.

To determine the cost of a rout we must add the euclidean distance (which is our free flow distance). This process is done in the function "route_cost".

And to determine if a route is feasible we will simulate arrival times using the congestion multipliers, service durations and possible waiting at customers with the times windows. To do so we will use the function : "evaluate_route".


In [ ]:
# The depot's latest allowed time defines the planning horizon.
horizon = float(instance["time_window"][0, 1])

# theta contains four boundaries, so it defines three traffic periods.
# Here we divide the planning horizon into three equal periods.
theta = np.linspace(0, horizon, 4)

# Normal traffic in period 1, congestion in period 2, normal in period 3.
# These are modelling assumptions, not data included in C101.txt.
rho = np.array([1.0, 1.5, 1.0])
traffic = {"theta": theta, "rho": rho}

def route_cost(route, instance):
    """This function returns the free-flow distance cost of one route, without traffic."""
    # D[i, j] is the Euclidean distance from node i to node j.
    distances = instance["edge_weight"]

    # Every route starts from the depot, node 0.
    previous = 0
    cost = 0.0

    # Add the distance to each customer, in the order of the route.
    for customer in route:
        cost += float(distances[previous, customer])
        previous = customer

    # Add the distance from the last customer back to the depot.
    cost += float(distances[previous, 0])
    return cost

def travel_arrival(i, j, ready_time, instance, traffic, fixed_departure=False):
    """Return (earliest arrival, chosen departure), allowing customer waiting."""
    distances = instance["edge_weight"]
    bounds = traffic["theta"]
    factors = traffic["rho"]
    H = float(instance["time_window"][0, 1])

    # A truck cannot begin a trip after the depot closing time.
    if ready_time >= H:
        return math.inf, math.inf

    # The first candidate is to depart immediately when ready.
    departures = [float(ready_time)]

    # At a customer, waiting until a new traffic period may be beneficial.
    # Example: leaving at the end of peak traffic may beat leaving immediately.
    if not fixed_departure:
        for boundary in bounds[1:-1]:
            if ready_time <= boundary < H:
                departures.append(float(boundary))

    best_arrival = math.inf
    best_departure = math.inf

    # Compare the arrival times obtained from all candidate departures.
    for departure in departures:
        # Find the traffic period in which the trip starts.
        slot = int(np.searchsorted(bounds[1:-1], departure, side="right"))

        # We keep the same congestion factor during the whole trip.
        travel_time = float(distances[i, j]) * float(factors[slot])
        arrival = departure + travel_time

        # Keep the earliest arrival within the planning horizon.
        if arrival <= H and arrival < best_arrival:
            best_arrival = arrival
            best_departure = departure

    return best_arrival, best_departure


def evaluate_route(route, instance, traffic):
    """Check capacity and all time windows for a route in its given order."""

    demands = instance["demand"]
    windows = instance["time_window"]
    service_times = instance["service_time"]
    capacity = float(instance["capacity"])
    H = float(windows[0, 1])

    # Mathematical assumption A5: departure from the depot is fixed at time 0.
    # The truck may wait only after serving a customer, not before leaving depot.
    current_node = 0
    ready_time = 0.0

    load = 0.0
    total_waiting = 0.0
    schedule = []

    for customer in route:
        # Prevent invalid customer numbers.
        if customer < 1 or customer >= len(demands):
            return {"feasible": False, "reason": "invalid customer"}

        # Increase the cumulative delivered load.
        load += float(demands[customer])
        if load > capacity:
            return {"feasible": False, "reason": "capacity exceeded"}

        # No waiting is allowed on the first trip from depot.
        # Later trips may wait at the previous customer.
        from_depot = current_node == 0
        arrival, departure = travel_arrival(
            current_node, customer, ready_time, instance, traffic,
            fixed_departure=from_depot
        )

        # If the customer is not open yet, wait before beginning service.
        service_start = max(arrival, float(windows[customer, 0]))

        # A service cannot start after its latest allowed time.
        if not math.isfinite(service_start) or service_start > windows[customer, 1]:
            return {"feasible": False, "reason": "customer time window"}

        # Waiting can occur after serving the previous customer,
        # or before beginning service at the current customer.
        total_waiting += departure - ready_time
        total_waiting += service_start - arrival

        # Once service is complete, the vehicle can depart or wait again.
        service_end = service_start + float(service_times[customer])
        schedule.append({
            "customer": customer,
            "arrival": arrival,
            "service_start": service_start,
            "service_end": service_end
        })
        current_node = customer
        ready_time = service_end

    # After the last customer, the truck must return to depot by time H.
    # A truck with no customers is not a useful route: handle it separately.
    if not route:
        return {"feasible": False, "reason": "empty route"}

    return_time, last_departure = travel_arrival(
        current_node, 0, ready_time, instance, traffic
    )

    if not math.isfinite(return_time) or return_time > H:
        return {"feasible": False, "reason": "late depot return"}

    total_waiting += last_departure - ready_time

    # Cost and duration are deliberately different measurements.
    return {
        "feasible": True,
        "distance": route_cost(route, instance),
        "duration": return_time,
        "waiting": total_waiting,
        "load": load,
        "return_time": return_time,
        "schedule": schedule
    }


def validate_solution(routes, instance, traffic):
    """Check the complete solution, including customer coverage and fleet size."""

    n = len(instance["demand"]) - 1

    # Each customer must appear once, in exactly one route.
    visited = [customer for route in routes for customer in route]
    if sorted(visited) != list(range(1, n + 1)):
        return False, "customers missing or duplicated"

    # One nonempty route corresponds to one vehicle.
    if any(len(route) == 0 for route in routes):
        return False, "empty route"
    if len(routes) > int(instance["vehicles"]):
        return False, "too many vehicles"

    # Check capacity, service windows, waiting and return time on each route.
    for route in routes:
        result = evaluate_route(route, instance, traffic)
        if not result["feasible"]:
            return False, result["reason"]

    return True, "all constraints respected"


def solution_cost(routes, instance, traffic):
    """Return total free-flow cost for a feasible solution; infinity otherwise."""

    # Congestion is used to check feasibility, not to price the arcs.
    feasible, reason = validate_solution(routes, instance, traffic)
    if not feasible:
        return math.inf
    return sum(route_cost(route, instance) for route in routes)

# Inspect one route and its timing (this is not the final solution).
example = evaluate_route([1], instance, traffic)
print("Single-customer route feasible:", example["feasible"])
if example["feasible"]:
    print("Free-flow route cost:", example["distance"])
    print("Congestion-aware return time:", example["return_time"])
    print("Waiting time:", example["waiting"])


### **2.4. Implementation of the Clark-Wright algorithm and the route elimination**

First Clarke-Wright Savings creates one route per customer and then considers merging routes. 